# Phytoplankton Phenology Study in the Northwest Atlantic Ocean

#### Import Python Libraries

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import cartopy
import pandas as pd
from shapely.ops import unary_union
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import Polygon
import rioxarray
import sys
sys.path.append(r'C:\Users\grace.davis\Documents\GitHub\RESOURCES\python')
import utilities
from utilities import get_prod_files
import cmocean
from matplotlib.colors import LogNorm
import cartopy.crs as crs
import statsmodels as sm
from statsmodels import nonparametric
from statsmodels.nonparametric import smoothers_lowess
import scipy
from scipy import signal
from scipy.signal import find_peaks
from scipy.signal import argrelextrema
from scipy.signal import savgol_filter
from scipy.integrate import trapezoid
from collections import Counter
import plotly.express as px
import seaborn as sns
import calendar
import plotly.graph_objects as go
import plotly.colors
from scipy import stats

In [2]:
daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
MABS = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABS_smoother_tests.zarr')
MABN = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\MABN_smoother_tests.zarr')
GB = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GB_smoother_tests.zarr')
GOMW = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOMW_smoother_tests.zarr')
GOME = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\GOME_smoother_tests.zarr')

In [3]:
shapefile = gpd.read_file('https://github.com/hsynan/READ-EDAB-Synan_hydrographic_climatologies/raw/refs/heads/main/data/shapefiles/NES_5REGIONS.zip')
MAB_south_loc = shapefile.iloc[[0]].set_crs("epsg:4326", inplace=True)
MAB_north_loc = shapefile.iloc[[1]].set_crs("epsg:4326", inplace=True)
GB_whole_loc = shapefile.iloc[[2]].set_crs("epsg:4326", inplace=True)
GOM_west_loc = shapefile.iloc[[3]].set_crs("epsg:4326", inplace=True)
GOM_east_loc = shapefile.iloc[[4]].set_crs("epsg:4326", inplace=True)
NES = shapefile.dissolve()

## Part 1: Bloom Detection Method Testing

### Threshold Method

#### Function for determining the climatological threshold value

In [4]:
def threshold_value(thld = 0.1, path = None):
    """
    Calculates the threshold value for chlorophyll-a based on a median baseline provided by the regional climatology.

    If no file path is provided, the path defaults to grabbing and reading the annual climatology file for the Northeast Shelf (NES) region. 
    The threshold is calculated by finding the percentage above the climatological CHL median for each pixel in the region.

    Args:
        thld (float, optional): The fraction value of the percentage above the median. This value defaults to 0.1 (10%).
        path (str, optional): The path to netCDF file used to calculate the threshold value. Defaults to None.

    Returns:
        xarray.DataArray: A spatial array containing the threshold values for each coordinate based on the median CHL value
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL') #finds path for the annual climatology file
        base_ds = xr.open_dataset(file[0]) #opens netCDF file
    else:
        base_ds = xr.open_dataset(path)
    median_CHL = base_ds.CHL_median #grabs CHL_median variable
    thld_value = median_CHL*(1+thld) #Sets threshold value per latitude and longitude point
    return thld_value

In [5]:
def spatial_threshold_value(shapefile_geometry,thld=0.1,path=None):
    """
    Creates a threshold value for a spatially averaged area.

    Using the threshold value, this function creates a spatially averaged threshold value for a region for use in other analysis.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        thld (float, optional): Percentage for threshold calcultion. Defaults to 0.1.
        path (str, optional): Path to climatology file. Defaults to None.
    
    Returns:
        float. Value of the regionally averaged chlorophyll threshold.
    """
    threshold = threshold_value(thld=thld,path=path)
    threshold.rio.write_crs("EPSG:4326",inplace=True)
    threshold.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    shapefile = shapefile_geometry.to_crs("EPSG:4326")
    clipped_thld = threshold.rio.clip(shapefile.geometry.apply(mapping), shapefile.crs, drop=True)
    clipped_thld = clipped_thld.mean(dim=['lat','lon'])
    clipped_thld = clipped_thld.item()
    return clipped_thld

In [6]:
MABS_clipped_thld = spatial_threshold_value(MAB_south_loc,thld=0.1)
MABN_clipped_thld = spatial_threshold_value(MAB_north_loc,thld=0.1)
GB_clipped_thld = spatial_threshold_value(GB_whole_loc,thld=0.1)
GOMW_clipped_thld = spatial_threshold_value(GOM_west_loc,thld=0.1)
GOME_clipped_thld = spatial_threshold_value(GOM_east_loc,thld=0.1)
MABS_clipped_med = spatial_threshold_value(MAB_south_loc,thld=0)
MABN_clipped_med = spatial_threshold_value(MAB_north_loc,thld=0)
GB_clipped_med = spatial_threshold_value(GB_whole_loc,thld=0)
GOMW_clipped_med = spatial_threshold_value(GOM_west_loc,thld=0)
GOME_clipped_med = spatial_threshold_value(GOM_east_loc,thld=0)

📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_CLIMATOLOGY\CHL
📦 Found 1 .nc files in: C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.

In [7]:
def bloom_mask(path=None,thld=0.1,clim_path=None,Boolean=False): #Produces True and False values
    """
    Creates a mask on the chlorophyll-a data to only include values above the threshold set by the climatological median.

    If no file path is provided, the function searches for all D8 files (8 day rolling mean) for the Northeast Shelf. 
    If a file path is provided, it is currently set to open zarr files. The code exists to open netCDFs as well, it just needs to be uncommented. 
    If no climatology file path is provided, the function searches for the annual climatology file of the Northeast Shelf region.
    The median chlorophyll-a values are then extracted and compared to the threshold value found from the regional climatology.
    If the median chlorophyll-a values exceed the threshold, the value is stored as true. Otherwise, it is stored as false.

    Args:
        path (str, optional): Path to the daily data (or other temporal resolution data). Defaults to None
        thld (float, optional): Fractional value for percentage to calculate the climatological threshold per coordinate. Defaults to 0.1 (10%)
        clim_path (str, optional): Path to regional climatology file. Defaults to None
        Boolean (bool, optional): Determines how values are stored. False keeps the actual values and marks false values as 0. True stores an array of True and False values. Defaults to False

    Returns:
        xarray.DataArray: A spatial array with Boolean values or floats for each coordinate based on the threshold value.
    """
    if path is None:
        files = get_prod_files('CHL',mapping='NES',period='D8') #Finds all 8 day rolling mean files for NES
        ds = xr.open_mfdataset(files)
    else:
        #ds = xr.open_mfdataset(path)
        ds = xr.open_zarr(path)
    if Boolean is True:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        is_bloom_CHL = med_CHL > clim_med #Is the median chl-a in each files greater than the climatological mean? Creates a Boolean array of trues and falses.
    else:
        med_CHL = ds.CHL_median #Extracts CHL_median variable for the files
        clim_med = threshold_value(thld=thld,path=clim_path)
        clim_med_new = clim_med.isel(time=0, drop=True) #Removes time dimension from climatological mean
        is_bloom_CHL = med_CHL.where(med_CHL > clim_med_new, 0) #Turns false into 0 and trues retain their value
    return is_bloom_CHL

In [8]:
def hist_local_chl(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the data to a specified box for more precise spatial analysis.

    If no file path is provided, the function opens all D8 files for the Northeast Shelf. Currently it opens the D8_combined zarr file but can be changed to open the netCDF files.
    This function takes the daily data and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the daily data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median values for the spatial averaged area for the full time series of the data.
    """
    if path is None:
        #daily_data = xr.open_mfdataset(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8*.nc')
        daily_data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_COMBINED.zarr',consolidated=True)
    else:
        daily_data = xr.open_mfdataset(path)
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    daily_data_local = daily_data.CHL_median.sel(#Clips the CHL_median data to the specified spatial bounds
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    daily_data_local = daily_data_local.mean(dim=['lat','lon']) #Averages the data over the spatial bounds
    return daily_data_local

In [9]:
def hist_clim_local(lat_min,lat_max,lon_min,lon_max,path=None):
    """
    Clips the climatology data to a specified box for more precise spatial analysis.

    If no file path is provided, the function searches for the annual climatology file for the Northeast Shelf.
    This function takes the median chlorophyll-a of the climatology and slices it to include the spatial data between the specified latitude and longitude values. 
    It then averages the chlorophyll-a median data over the area specified, for use in plotting.

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default
        path (str, optional): The path to the climatology data. Defaults to None

    Returns:
        xarray.DataArray: Chlorophyll-a median value for the spatial averaged area for the climatology.
    """
    if path is None:
        file = get_prod_files('CHL',mapping='NES',period='ANNUAL')
        clim = xr.open_dataset(file[0])
    else:
        clim = xr.open_dataset(path)
    clim_med = clim.CHL_median
    lat_min = lat_min
    lat_max = lat_max
    lon_min = lon_min
    lon_max = lon_max
    clim_med = clim_med.sel(
    lat=slice(lat_min,lat_max),
    lon=slice(lon_min,lon_max)
    )
    clim_med_bounded = clim_med.mean(dim=['lat','lon'])
    return clim_med_bounded


In [10]:
def bound_local(lat_min,lat_max,lon_min,lon_max): #Builds polygon shape for the map
    """
    Creates a polygon shape for mapping

    This function takes in boundary coordinates and makes a shape to be used for plotting. 

    Args:
        lat_min (float, required): Minimum latitude value of box (southern boundary). No default
        lat_max (float, required): Maximum latitude value of box (northern boundary). No default
        lon_min (float, required): Minimum longitude value of box (western boundary). No default
        lon_max (float, required): Maximum longitude value of box (eastern boundary). No default

    Returns: 
        POLYGON
    """
    coords = [(lon_min,lat_min),
              (lon_max,lat_min),
              (lon_max,lat_max),
              (lon_min,lat_max),
              (lon_min,lat_min)]
    box_polygon = Polygon(coords)
    return box_polygon

In [11]:
def percent_above_thld(data,clipped_thld):
    """
    Calculates the percentage of data points that lie above a threshold value.
    
    The data provided must be clipped to a region prior to inputting into function. Otherwise the function will run it for the entire spatial data in the dataset.

    Args: 
        data (xarray.Dataset, required): Dataset for analysis. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for region

    Returns:
        float. The percentage of datapoints that lie above the threshold.
    """
    dataset = data['CHL_median']
    threshold = clipped_thld
    total_above = int((dataset>threshold).sum())
    percent = (total_above/len(data))*100
    return percent

In [12]:
def percent_deviation(dataset,clipped_median,clipped_thld):
    """
    Calculates the amount of data that is a certain percentage deviated from tmedian.

    Args:
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        clipped_median (float, required): The pre-calculated median for the region. No defaults
        clipped_thld (float, required): The pre-calculated threshold for the region. No defaults

    """
    
    top = dataset.squeeze()-clipped_median
    fraction = top/clipped_thld
    percent_dev = fraction*100
    return percent_dev

### Rate of Change

In [13]:
def bounding_data(dataset,shapefile_geometry):
    """
    Regionally subsets a dataset for general analysis.

    This function takes a shapefile geometry and subsets a larger dataset to only include data within the shapefile. The data is averaged along the lat and lon dimensions.

    Args:
        dataset (xarray.Dataset, required): General dataset in question. No defaults
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults

    Returns:
        xarray.DataArray. The arrays of spatially sliced data.
    """
    dataset.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=True)
    dataset.rio.write_crs("epsg:4326", inplace=True)
    clipped_daily = dataset.rio.clip(shapefile_geometry.geometry.apply(mapping), shapefile.crs, drop=True)
    regional_year = clipped_daily.CHL_median.mean(dim=['lat','lon'])
    return regional_year

In [14]:
def smoothing_data(shapefile_geometry=None,dataset = None, path=None,method="SavGol",window=15,poly=3,deriv=0,frac=0.00117):
    """
    Smoothes the raw chlorphyll-a data using a specific smoothing technique.

    If no method is provided, the default is the Savistky-Golay technique which has default parameters of a 15 day window and a polyorder of 3.
    If method is provided as "lowess", the frac value defaults to 0.00117, equivalent of a 12 day window on a 27 year time series.
    If no dataset path is provided, the function searches for D8 CHL files for the NES region. Currently, it opens the zarr file, but can be uncomment to open netCDFs.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        dataset (xarray.Dataset, optional): A spatially averaged dataset. Default is None
        path (str, optional): Path to a dataset. Defaults to daily D8 data for the full time series.
        method (str, optional): Smoothing technique applied. Defaults to "SavGol" but can also receive "lowess".
        window (int, optional): Window for SavGol smoothing. Default is 15
        poly (int, optional): polyorder for SavGol smoothing. Default is 3
        deriv (int, optional): Derivative of SavGol function. 0 provides smoothed data and 1 provides the first derivative. Defaults to 0 
        frac (float, optional): Frac value for lowess smoothing. Only necessary for using lowess smoothing. Default is 0.00117

    Returns:
        numpy.ndarray. Array of smoothed chlorophyll data values.
    """
    if dataset is not None:
        data = dataset
    elif path is None:
        #file = get_prod_files('CHL',map_region='NES',period='D8')
        #data = xr.open_mfdataset(file)
        data = xr.open_zarr(r'C:\Users\grace.davis\Documents\GitHub\DATASETS\OCCCI\V6.0\PRODUCTS\NES_4KM_DAY8\CHL\D8_combined.zarr')
    else:
        data = xr.open_mfdataset(path)
    
    chl = data['CHL_median']
    if method == "SavGol":
        time = data.time.astype('int64') #Changes time values to integers for smoothing
        median = median["CHL_median"].values
        mask = ~np.isnan(median)
        chl_interp = pd.Series(median).interpolate(method='linear').bfill().ffill().values #Linear interpolation to fill in NaN values
        sg_smoothed = savgol_filter(chl_interp,window_length=window,polyorder=poly,deriv=deriv) #Applying the SavGol filter
        sg_smoothed_full = np.copy(sg_smoothed)
        sg_smoothed_full[~mask]=np.nan #Puts the NaN values back in after smoothing
        smoothed_median = sg_smoothed_full[~mask]
    elif method == "lowess":
        time = dataset.time.astype('int64') 
        if shapefile_geometry is None:
            print("Shapefile required for smoothing")
        dataset = bounding_data(dataset,shapefile_geometry)
        median=dataset.to_dataframe() 
        median=median['CHL_median']
        smoothed_median = sm.nonparametric.smoothers_lowess.lowess(median,time,frac=frac) #Applying the lowess filter
    else:
        print("Error: Must specify smoothing technique")
    return smoothed_median

This function finds start and end dates of blooms

In [15]:
def bloom_peak_detection(shapefile_geometry,clipped_thld,dataset=None,window_for_peak=10,days=14,prm=0.1,**kwargs):
    """
    Detects all peak chlorophyll values that exceed the climatological threshold

    This function uses the smoothed chlorophyll data, identified peak values, and then masks that data to include only peaks that exceed the threshold set by the climatology.
    This function uses the find_peaks function from scipy, as well as the threshold_value() function and smoothing_data() function. 
    If no dataset is provided, the function smooths data based on the smoothing_data() function.
    The clipped_thld variable is created in the rolling_peak_window function or as a global variable.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold value for the region of interest. No defaults
        dataset (xarray.Dataset, optional): Already smoothed dataset. Defaults to None
        days (int, optional): Distance variable for scipy find_peaks. Distance allowed between consecutive peaks. Default is 14
        prm (float, optional): Prominence variable for find_peaks. Percent above the other peaks to be considered a peak. Default is 0.015
        **kwargs: Keywords for underlying functions. Expected keyword arguments include:
            - path (str, optional): Path to the data. Defaults to daily D8 data for the full time series.
            - method (str, optional): Smoothing technique. Defaults to "SavGol".
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only necessary if method == "lowess". Defaults to 0.00117
    
    Returns:
        List. List of days since the start of the dataset where the chlorophyll peaked and was above the threshold.
    """
    # STEP 1: Define the dataset. Uses a smoothed dataset (if provided). Else, it smooths the raw data provided for the region of interest.
    if dataset is None:
        smoothed_CHL = smoothing_data(shapefile_geometry,**kwargs)
    else:
        smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']

    # STEP 2: Find chlorophyll peaks with find peaks function.
        # Default of 21 days for distance was chosen after testing distances from 10-31. 10-20 separated peaks that never crossed below the threshold.
        # Default prominence of 0.1 captures major blooms while ignoring small peaks from daily fluctuations/sensor noise. Tested values in range of 0.01 - 0.2. 
    chl_peak_loc, _ = find_peaks(smoothed_CHL,distance=days,prominence=prm) #Finds all peaks
    chl_peaks = []
    chl_series = pd.Series(smoothed_CHL.values) #Turns chlorophyll values into a pandas series

    # STEP 3: Create a Boolean list for values that surpass/do not exceed the set threshold.
    is_above_threshold = chl_series>clipped_thld #Creates a true and false list. True if the value exceeds the threshold.

    # STEP 4: Searches Boolean list for places where the value switches from True to False (or False to True)
    change_from_prev_day = is_above_threshold != is_above_threshold.shift() #Checks if there is a change from previous day

    # STEP 5: Create streak IDs for each event and group events with the same ID together
    streak_IDs = change_from_prev_day.cumsum() #Creates ID for each event (New ID starts when the Boolean value changes. If no change, the ID is the same for that day)
    streak_lengths = is_above_threshold.groupby(streak_IDs).transform('sum') #Groups events together with the same ID and calculates the number of days that share that ID
    
    # STEP 6: Check to ensure the peak is above the threshold and check to see if its streak ID is >= to the defined window_for_peak.
        # If both conditions are true, we add it to the chl_peaks list. If one or both is not met, the peak is discarded.
    for peak in chl_peak_loc:
        peak_above_threshold = is_above_threshold[peak]
        peak_length = streak_lengths[peak]>=window_for_peak
        if peak_above_threshold and peak_length:
            chl_peaks.append(peak)
    return chl_peaks

In [16]:
def bloom_event_detection(shapefile_geometry,clipped_thld,dataset=None,event_distance=21,peak_window=10,verbose=False,**kwargs):
    """
    This function finds peaks in the chlorophyll-a time series and then groups together peaks in the same event based on proximity.

    If no dataset is provided, the function smooths out the data in the path given or the default data in the smoothing_data function.
    The function starts with finding all days since the start of the time series where chlorophyll-a concentrations peaked. 
    It then finds the threshold for comparison later. If there are no peaks, the function returns an empty list.
    A ten day rolling window is created for each peak to see if the chlorophyll value drop below the climatological threshold. If it does, the loop breaks.
    If peaks are too close together or the chlorophyll value does not drop below the threshold, they are considered one event. If these conditions are not met, they are separate events.

    Args:
        shapefile_geometry (variable, required): Shapefile for region in question. No defaults
        clipped_thld (variable, required): Threshold value for the region of interest. No defaults
        dataset (variable, optional): Already smoothed dataset. Default is None
        event_distance (int, optional): The number of days peaks must be apart to be considered separate events. Defaults to 21
        peak_window (int, optional): The number of days the chlorophyll concentration must remain above or below the threshold. Defaults to 10
        **kwargs: Additional arguments for bloom_peak_detection function. Inputs could include:
            - days (int, optional): Distance for find_peaks function. Defaults to 14
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.1
            - method (str, optional): Smoothing method for smoothing_data function. Defaults to "SavGol"
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): Polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only need if method == "lowess". Defaults to 0.00117
    Returns: 
        List: List of bloom event and peaks within each event.
    """
    # STEP 1: Finding peaks and the threshold value
    chl_peaks = bloom_peak_detection(shapefile_geometry,dataset=dataset,clipped_thld=clipped_thld,**kwargs)
    if not chl_peaks: #Returns empty list if no peaks were found
        return []

    # STEP 2: Defines the smoothed dataset
    if dataset is None:
        smoothed_CHL = smoothing_data(shapefile_geometry,**kwargs)
    else:
        smoothed_CHL = dataset['smoothed_sg_win_15_poly_3']
        smoothed_CHL = smoothed_CHL.values
    bloom_events = []

    # STEP 3: Creates the range for chlorophyll values to be observed in and identifies peak timeline
    current_event = [chl_peaks[0]] #Current event starts at the first peak identified
    days_between_events = event_distance #The number of days that must pass between conditions for the peaks to be considered separate events
    for i in range(1,len(chl_peaks)):
        previous_peak = chl_peaks[i-1] #Finds the previous peak
        current_peak = chl_peaks[i]
        chl_between_peaks = smoothed_CHL[previous_peak:current_peak] #Creates a list of all chlorophyll values between the current peak and previous peak
        dropped_below_thld = False

    # STEP 4: Find if the chlorophyll concentration drops below the threshold for a certain number of consecutive days
        #Checks to see if the number of days between chlorophyll peaks is above the specified peak window
        if len(chl_between_peaks)>=peak_window:
            chl_series = pd.Series(chl_between_peaks)
            #If all chlorophyll values are below the pre-determined threshold, dropped_below_thld is true. It adds up the trues and falses and finds the spots where the value is equal to peak_window
            dropped_below_thld = (chl_series<clipped_thld).rolling(window=peak_window).sum().eq(peak_window).any()
        if verbose is True:
            print(f"\n--- Checking transition from peak at index {previous_peak} to {current_peak} ---")
            print(f"Distance between peaks: {current_peak - previous_peak} (Needs to be >= {days_between_events} to split based on time)")
            print(f"Number of data points in gap: {len(chl_between_peaks)}")
            print(f"Minimum CHL value in gap: {min(chl_between_peaks):.3f} (Threshold is {clipped_thld:.3f})")
            print(f"Did it stay below threshold for {peak_window} consecutive points? {dropped_below_thld}")
    # STEP 5: Append events to events list. 
        if current_peak-previous_peak<days_between_events or not dropped_below_thld: #If peaks are too close together or does not drops below threshold, they are the same event.
            current_event.append(current_peak)
        else: #Peaks are an appropriate distance apart or chl drop below the threshold.
            bloom_events.append(current_event)
            current_event = [current_peak]
    bloom_events.append(current_event)
    return bloom_events


In [17]:
def rolling_peak_window(shapefile_geometry,clipped_thld,clipped_med,dataset=None,init_term_window=5,**kwargs):
    """
    This function finds the initiation and termination dates of blooms based on a rolling peak window.

    The function finds the climatological threshold value for the region and the rate of change for all data points.
    Then it defines the peak windows based on the first peak in the event.
    Using the rate of change, it finds the initiation date. This is either where the rate of change is consecutively positive or where the previous bloom terminates, whichever comes first.
    Then it finds the termination date. This happens once the rate of change has been decreasing for a specified time and it has dropped below the threshold value.
    It then merges peaks with the same termination date.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults
        dataset (xarray.Dataset, optional): Already smoothed dataset. Defaults to None
        init_term_window (int, optional): Amount of time each condition must be met for it to trigger an initiation or termination date. Defaults to 5
        **kwargs: Additional input for bloom_event_detection and threshold_value functions. Possible inputs include: 
            - peak_window (int, optional): The amount of time a peak must remain above the threshold for it to be considered an event. Defaults to 10  
            - days (int, optional): Distance for find_peaks function. Defaults to 10
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.015
            - method (str, optional): Smoothing method for smoothing_data function. Defaults to "SavGol"
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): Polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only need if method == "lowess". Defaults to 0.00117
    Returns:
        tuple: A tuple containing (start_DOY, end_DOY, merge_bloom_events), where:
            start_DOY (list): The list of bloom initiation days.
            end_DOY (list): The list of bloom termination days.
            merge_bloom_events (list): The list of bloom events.
    """
    # STEP 1: Find the climatological median, rate of change, and bloom events
    chl_median = dataset['smoothed_sg_win_15_poly_3'].to_series().reset_index(drop=True)
    roc = dataset['ROC_SG'].to_series().reset_index(drop=True)
    bloom_events = bloom_event_detection(shapefile_geometry,dataset=dataset,clipped_thld=clipped_thld,**kwargs)
    
    # STEP 2: Define peak windows
    last_end_day, last_start_day = 0, 0
    start_DOY, end_DOY, merge_bloom_events, max_roc_indices = [], [], [], []
    max_index = len(roc)-1

    # STEP 3: Identify all possible initiation and termination dates
    is_roc_negative =  roc < 0
    is_roc_positive = roc >= 0
    is_below_threshold = chl_median < clipped_thld
    is_below_median = chl_median <= clipped_med

    #Find all days for time series where initiation conditions are met (positive growth and below the threshold)
    #initiation_conditions_met = is_roc_positive & is_below_threshold
    initiation_conditions_met = is_roc_positive & is_below_threshold
    initiation_rolling = initiation_conditions_met.rolling(window=init_term_window).sum()
    initiation_days = initiation_rolling[initiation_rolling == init_term_window].index.to_numpy()

    #Find all days for time series where termination conditions are met
    termination_conditions_met = is_roc_negative & is_below_threshold
    termination_rolling = termination_conditions_met.rolling(window=init_term_window).sum()
    termination_days = termination_rolling[termination_rolling == init_term_window].index.to_numpy()

    #Find all local troughs for the full dataset. The troughs must have chl values lower than the three consecutive days on either side. 
    #This smooths out some of the smaller bumps caused by the noisy chl-a data and keeps major troughs. Tested 1,2,3. 
    local_minimum = (chl_median < chl_median.shift(3)) & (chl_median < chl_median.shift(-3)) & is_below_median
    local_minimum = local_minimum[local_minimum].index.to_numpy()

    # STEP 4: Find the initiation date of the bloom based on the rate of change.
    for event in bloom_events:
        event_start = event[0]
        event_end = event[-1]
        end_of_window = min(max_index,event[-1]+180) #Sets the end of the window to be 180 days from the last peak in the event or the end of the dataset, whichever comes first.
        start_day = last_end_day #Sets the start of the window to the last end day
        possible_init_dates = initiation_days[(initiation_days < event_start)&(initiation_days >= start_day)] #Find all possible initiation days between the end of the last bloom and the first peak in the current event.

        if len(possible_init_dates) > 0 and last_end_day >= last_start_day: #Ensures that the initiation date is not before the previous bloom's termination date
            possible_start_day = possible_init_dates[-1] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= last_end_day) & (local_minimum <= event_start)] #Finds all troughs between first peak and the previous termination
            if len(window_troughs) > 0:
                 closest_trough = np.abs(window_troughs - possible_start_day).argmin() #Finds the closest trough to the first peak
                 start_day = window_troughs[closest_trough] 
            else:
                 start_day = possible_start_day  

    # STEP 5: Find the termination date 
        end_day = event_end
        possible_term_dates = termination_days[(termination_days>event_end) & (termination_days<=end_of_window)]

        if len(possible_term_dates)>0:
            possible_end_day = possible_term_dates[0] - init_term_window + 1
            window_troughs = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
            if len(window_troughs) > 0:
                closest_trough = np.abs(window_troughs - possible_end_day).argmin() #Finds the closest trough to the first peak
                end_day = window_troughs[closest_trough]

        else: #If termination conditions are not met for a bloom, find the next trough that is below the threshold value and make that the termination date.
                potential_trough = local_minimum[(local_minimum >= event_end) & (local_minimum <= end_of_window)]
                if len(potential_trough) > 0:
                    end_day = potential_trough[0]

    # STEP 6: Merge and/or append events to the lists
        same_bloom = len(start_DOY) > 0 and start_day == start_DOY[-1] and end_day == end_DOY[-1]
        same_timing = (last_end_day>0) and (event[0]<=last_end_day)
        if same_bloom or same_timing: #This ensures that termination dates are not duplicated and every initiation date has a termination date
            merge_bloom_events[-1].extend(event)
            if end_day>end_DOY[-1]:
                 end_DOY[-1] = end_day
        else:
            start_DOY.append(start_day)
            end_DOY.append(end_day)
            merge_bloom_events.append(list(event))
        last_start_day = start_day #Resets start and end dates for the loop
        last_end_day = end_day
    return start_DOY,end_DOY,merge_bloom_events

In [18]:
def max_peaks(shapefile_geometry, dataset, clipped_thld, clipped_med):
    """
    Finds the peak in an event with the maximum chlorophyll concentration for the event.
    Uses the rolling_peak_window() function.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        dataset (xarray.Dataset, required): The dataset for analysis
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults

    Returns:
        List. A list of peaks associated with that blooms maximum chlorophyll concentration.
    """
    bloom_events = rolling_peak_window(shapefile_geometry,dataset=dataset,clipped_thld=clipped_thld, clipped_med=clipped_med)[2]
    peak_DOY_time_series = []
    peak_dates_time_series = []
    peak_index = []
    region_smoothed = dataset['smoothed_sg_win_15_poly_3']
    region_smoothed_values = region_smoothed.values
    region_time_smoothed = dataset['time'].values
    for event in bloom_events:
        peak_chl_values = -float('inf')
        max_chl_day = None
        flatten_event = []
        for item in event:
            if isinstance(item,(tuple,list,np.ndarray)):
                flatten_event.extend(item)
            else:
                flatten_event.append(item)
        for day in flatten_event:
            chl_peak = region_smoothed_values[int(day)]
            if chl_peak>peak_chl_values:
                peak_chl_values = chl_peak
                max_chl_day = day
        peak_DOY = region_time_smoothed[max_chl_day]
        peak_chl = region_smoothed.values[max_chl_day]
        ts = pd.Timestamp(peak_DOY)
        date = ts.date()
        doy = ts.dayofyear
        peak_dates_time_series.append(date)
        peak_DOY_time_series.append(doy)
        peak_index.append(peak_chl)
    return peak_DOY_time_series,peak_dates_time_series, peak_index

In [19]:
def max_roc_for_bloom(shapefile_geometry,clipped_thld,clipped_med,dataset=None,**kwargs):
    """
    Finds the maximum rate of change for each bloom.

    This function finds the initiation and termination dates for each bloom. Then it finds the rate of change for every data point. 
    It then finds the maximum rate of change between the initiation and termination date and then adds it to the start day value to get the DOY value for the maximum rate of change.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        clipped_med (float, required): Pre-calculated climatological median for the region. No defaults
        dataset (xarray.Dataset, optional): The zarr file with the data. Defaults to None
        **kwargs: Additional variables for internal functions. Inputs could include:
            - days (int, optional): Distance for find_peaks function. Defaults to 10
            - prm (float, optional): Prominence for find_peaks function. Defaults to 0.015
            - method (str, optional): Smoothing method for smoothing_data function. Defaults to "SavGol"
            - window (int, optional): Window for SavGol smoothing. Defaults to 15
            - poly (int, optional): Polyorder for SavGol smoothing. Defaults to 3
            - frac (float, optional): Frac value for lowess smoothing. Only need if method == "lowess". Defaults to 0.00117
    Returns:
        tuple: A tuple containing (max_roc, start_day_bloom, end_day_bloom, bloom_peaks), where:
            max_roc (list): The list of maximum rates of change identified.
            start_day_bloom (list): The list of bloom initiation days.
            end_day_bloom (list): The list of bloom termination days.
            bloom_peaks (list): The list of bloom events.
    """
    start_day_bloom, end_day_bloom, bloom_peaks = rolling_peak_window(shapefile_geometry=shapefile_geometry,dataset=dataset,clipped_thld=clipped_thld,clipped_med=clipped_med,**kwargs)
    roc = dataset['ROC_SG'].values
    max_roc = []
    for i in range(len(start_day_bloom)):
        start_day=start_day_bloom[i]
        if i <len(end_day_bloom):
            end_day=end_day_bloom[i]
        else:
            continue
        range_roc = roc[start_day:end_day]
        if len(range_roc)>0:
            range_max_roc = np.nanargmax(range_roc) #Finds local maximum rate of change for each detected bloom
            max_roc_index = start_day+range_max_roc #Gets the actual day of year value
            max_roc.append(max_roc_index)
    return max_roc, start_day_bloom, end_day_bloom, bloom_peaks

## Part 2: Quantify the Number of Phytoplankton Bloom Days Per Year

In [20]:
def bloom_classification(shapefile_geometry,dataset,clipped_thld,clipped_med):
    """
    Classifies identified blooms by the peak DOY as spring, fall, or other.

    This function classifies a bloom as a spring bloom, fall bloom, or other bloom based on its peak DOY. Based on the seasons and relative start and peak times, the DOY ranges are
        - 1 to 59 for winter blooms (January 1 to February 28)
        - 61 to 152 for spring blooms (March 1 to June 1)
        - 244 to 366 for fall blooms (September 1 to December 31)
    Other blooms do not fall within these DOY ranges. 

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region in question. No defaults
        dataset (xarray.Dataset, required): Dataset of region of interest. No defaults
        clipped_thld (float, required): The pre-calculated climatological threshold for the region. No defaults

    Returns:
        Tuple: A tuple containing all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms, where:
            all_blooms: A list of all blooms as their string classification "Spring", "Fall", or "Other".
            spring_blooms: A list of all spring blooms for the dataset. Returns the peak DOY. 
            fall_blooms: A list of all fall blooms for the dataset. Returns the peak DOY.
            winter_blooms: A list of all winter blooms for the dataset. Returns the peak DOY
            other_blooms: A list of blooms not classified as spring or fall blooms. Returns peak DOY.
    """
    peak_DOY, _, _ = max_peaks(shapefile_geometry,dataset,clipped_thld=clipped_thld,clipped_med=clipped_med)
    spring_blooms = []
    fall_blooms = []
    winter_blooms = []
    other_blooms = []
    all_blooms = []
    for peak in peak_DOY:
        #Identify spring blooms as first bloom of the year or bloom within the spring DOY range
        doy = peak
        if doy >= 60 and doy <=152:
            potential_spring_bloom = peak
            spring_blooms.append(potential_spring_bloom)
            all_blooms.append("Spring")
        #Identify fall blooms as last bloom of the year or within the fall DOY range
        elif doy >= 245 and doy <=366:
            potential_fall_bloom = peak
            fall_blooms.append(potential_fall_bloom)
            all_blooms.append("Fall")
        #Identify other blooms as other
        elif doy >=1 and doy <=59:
            potential_winter_bloom = peak
            winter_blooms.append(potential_winter_bloom)
            all_blooms.append("Winter")
        else:
            potential_other_bloom = peak
            other_blooms.append(potential_other_bloom)
            all_blooms.append("Other")
    return all_blooms, spring_blooms, fall_blooms, winter_blooms, other_blooms

In [21]:
def bloom_days_per_year(dataset, year, start_DOY, end_DOY):
    """
    Calculates the number of bloom days per year from bloom initiation to termination.

    This function uses the start_DOY and end_DOY lists from the rolling_peak_window function.
    Then it slices the data and calculates all of the bloom days for the specified year.
    This function assumes a start date of January 1 if the start date falls in the previous year. 
    It assumes a termination date of December 31 if the termination date is in the following year.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function. Defaults to end_DOY

    Returns:
        Int. The number of bloom days for the given year.
    """
    number_bloom_days = []
    days_in_year = pd.to_datetime(f"{year}-12-31").dayofyear
    for x in range(len(start_DOY)):
        if x < len(end_DOY):
            bloom_start_date = int(start_DOY[x])
            start_date = pd.to_datetime(dataset['time'].values[bloom_start_date])
            if start_date.year == year:
                start_day = str(start_date.dayofyear)
            elif start_date.year<year:
                start_day = 1
            else:
                continue
            bloom_end_date = int(end_DOY[x])
            end_date = pd.to_datetime(dataset['time'].values[bloom_end_date])
            if end_date.year == year:
                end_day = str(end_date.dayofyear)
            elif end_date.year>year:
                end_day = days_in_year
            else:
                end_day = start_day
            amount_bloom_days = int(end_day)-int(start_day)
            number_bloom_days.append(amount_bloom_days)
        else:
            start_day = start_DOY[x]
            end_day = days_in_year
            amount_bloom_days = int(end_day)-int(start_day)+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_year=sum(number_bloom_days)
    return bloom_days_per_year

In [22]:
def bloom_days_per_month(dataset, year, month, start_DOY, end_DOY):
    """
    Calculates the number of bloom days per month from bloom initiation to termination.

    This function uses the rolling_peak_window() function. It finds the initiation and termination for each date in the time series.
    Then it slices the data and calculates all of the bloom days for the specified month.

    Args:
        dataset (xarray.Dataset, required): Dataset variable of smoothed data. No defaults
        year (int, required): The year for calculating the number of bloom days. No defaults
        month (int, required): The number of the month of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window_function. Defaults to end_DOy

    Returns:
        Int. The number of bloom days for the given year.
    """
    number_bloom_days = []
    _, last_day_month = calendar.monthrange(year,month)
    month_start = pd.Timestamp(year=year,month=month,day=1)
    month_end = pd.Timestamp(year=year, month=month,day=last_day_month)

    for x in range(len(start_DOY)):
        start_index = int(start_DOY[x])
        bloom_start_date = pd.to_datetime(dataset['time'].values[start_index])
        if x < len(end_DOY):
            end_index = int(end_DOY[x])
            bloom_end_date = pd.to_datetime(dataset['time'].values[end_index])
        else:
            bloom_end_date = pd.Timestamp(year=year,month=12,day=31)
        start_overlap = max(bloom_start_date,month_start)
        end_overlap = min(bloom_end_date,month_end)

        if start_overlap <= end_overlap:
            amount_bloom_days = (end_overlap-start_overlap).days+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_month=sum(number_bloom_days)
    return bloom_days_per_month

In [23]:
def bloom_days_above_threshold(clipped_thld, dataset, year, start_DOY, end_DOY, bloom_events):
    """
    Calculate the number of days in a year where the bloom days are considered days within the bloom range where the chlorophyll-a concentration is above the climatological threshold.

    This function finds the subset of days during a bloom where the chlorophyll concentration is above the predetermined climatological threshold.
    It requires the list outputs from the rolling_peak_window function and the float value from the spatial_threshold_value function.

    Args: 
        clipped_thld (float, required): Pre-calculated climatological threshold for the region of interest. No defaults
        dataset (xarray.Dataset, required): Dataset for region of interest. No defaults
        year (int, required): Year of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY
        bloom_events (list, required): List of bloom events pre-calculated from rolling_peak_window function. Defaults to blooms_events
    Returns:
        int: The integer number of days in the year the chlorophyll is above the threshold during a bloom.
    """
    chl_median = dataset['smoothed_sg_win_15_poly_3'].values
    times = pd.to_datetime(dataset['time'].values)

    start_at_thld_list = []
    end_at_thld_list = []
    for i in range(len(start_DOY)):
        start = start_DOY[i]
        peak = bloom_events[i][0]
        for j in range(start+1,peak,1):
            if chl_median[j] >= clipped_thld:
                start_at_thld = j
                start_at_thld_list.append(start_at_thld)
                break

        end = end_DOY[i]
        peak = bloom_events[i][-1]
        for j in range (end-1,peak,-1):
            if chl_median[j] >= clipped_thld:
                end_at_thld = j
                end_at_thld_list.append(end_at_thld)
                break
    number_bloom_days = []
    days_in_year = pd.to_datetime(f"{year}-12-31").dayofyear
    for x in range(len(start_at_thld_list)):
        if x < len(end_at_thld_list):
            bloom_start_date = int(start_at_thld_list[x])
            start_date = times[bloom_start_date]
            if start_date.year == year:
                start_day = str(start_date.dayofyear)
            elif start_date.year<year:
                start_day = 1
            else:
                continue
            bloom_end_date = int(end_at_thld_list[x])
            end_date = times[bloom_end_date]
            if end_date.year == year:
                end_day = str(end_date.dayofyear)
            elif end_date.year>year:
                end_day = days_in_year
            else:
                end_day = start_day
            amount_bloom_days = int(end_day)-int(start_day)
            number_bloom_days.append(amount_bloom_days)
        else:
            start_day = start_at_thld_list[x]
            end_day = days_in_year
            amount_bloom_days = int(end_day)-int(start_day)+1
            number_bloom_days.append(amount_bloom_days)

    bloom_days_per_year=sum(number_bloom_days)
    return bloom_days_per_year

In [24]:
def annual_events(shapefile_geometry,dataset,clipped_thld,first_year=1998,last_year=2026):
    """
    Finds the number of blooms per year for the dataset.

    This function categorizes blooms into years based on their peak date. It uses the max_peaks function.

    Args:
        shapefile_geometry (geopandas.GeoDataFrame, required): Shapefile of the region of interest. No defaults
        dataset (xarray.Dataset, required): Dataset of the region of interest. No defaults
        clipped_thld (float, required): Pre-calculated climatological threshold for the region. No defaults
        first_year (int, optional): First year in the time series of interest. Defaults to 1998.
        last_year (int, optional): Year after the last year of interest in the time series. Defaults to 2026

    Returns:
        Dictionary: A dictionary of the year and the number of events in that year.
    """
    # Peak date and DOY
    _, peak_date ,_= max_peaks(shapefile_geometry,dataset,clipped_thld)

    #Year
    peak_year = [date.year for date in peak_date]
    blooms_per_year = Counter(peak_year)
    blooms_per_year = {year: blooms_per_year.get(year,0) for year in range (int(first_year),int(last_year))}
    return blooms_per_year

In [25]:
def bloom_duration(bloom_index,start_DOY,end_DOY):
    """
    Finds the length of each bloom event.

    Given the bloom index (1 to the length of start_DOY), the start DOY is subtracted from the end DOY to get the total duration of the bloom.

    Args: 
        bloom_index : Bloom index of interest. Values range from 1 to len(start_DOY) + 1. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. No defaults
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. No defaults

    Returns:
        int. Integer of the number of days of the bloom
    """
    start_date = start_DOY[bloom_index]
    end_date = end_DOY[bloom_index]
    duration = end_date-start_date
    return duration

In [26]:
def thld_bloom_duration(clipped_thld,dataset,start_DOY,end_DOY,bloom_events):
    """
    Calculates the amount of time a bloom spends above the climatological threshold.

    This function first calculates the climatological median threshold for the region of interest.
    It then the first day after the established start of the bloom that crosses the determined threshold.
    Then it finds the last day it crosses the threshold before the bloom terminates. It uses these dates to find the duration above the threshold.

    Args: 
        clipped_thld (float, required): The pre-calculated climatological threshold for the region of interest. No defaults
        dataset (xarray.Dataset, required): The dataset of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY
        bloom_events (list, required): List of bloom events pre-calculated from rolling_peak_window function. Defaults to blooms_events


    Returns:
        Tuple: A tuple including bloom_duration_thld, start_thld, and end_thld, where:
            bloom_duration_thld: List of durations above the threshold for the full dataset. List of integers
            start_thld: List of first days surpassing the threshold for the full dataset. List of integers
            end_thld: List of last days crossing the threshold for the full dataset. List of integers.
    """
    chl_median = dataset['smoothed_sg_win_15_poly_3'].interpolate_na(dim='time',method='linear')

    bloom_duration_thld = []
    start_thld = []
    end_thld = []
    for i in range(len(start_DOY)):
        start = start_DOY[i]
        end = end_DOY[i]
        peak_start = bloom_events[i][0]

        start_at_thld = None
        end_at_thld = None
        for j in range(start+1,peak_start+1,1):
            if chl_median[j] >= clipped_thld:
                start_at_thld = j
                break

        peak_end = bloom_events[i][-1]
        for j in range (end-1,peak_end-1,-1):
            if chl_median[j] >= clipped_thld:
                end_at_thld = j
                break
        if start_at_thld is not None:
            if end_at_thld is None:
                end_at_thld = len(chl_median)-1
            bloom_days_thld = end_at_thld-start_at_thld
            bloom_duration_thld.append(bloom_days_thld)
            start_thld.append(start_at_thld)
            end_thld.append(end_at_thld)
        else:
            bloom_duration_thld.append(None)
            start_thld.append(None)
            end_thld.append(None)
    return bloom_duration_thld, start_thld, end_thld

#### Integrated chlorophyll a

In [27]:
def event_integrated_chla(dataset,bloom_index,start_DOY,end_DOY):
    """
    Calculates the integrated chlorophyll-a concentration per bloom.

    This function uses trapezoid integration to estimate the amount of chlorophyll-a per bloom.
    It includes all chl-a values (0 to maximum value for the peak).
    The bounds of integration are determined by the start and end date found from the rolling_peak_window function. This function must be run prior to this one and use start_DOY and end_DOY as variables.

    Args:
        dataset (xarray.Dataset, required): The raw regional dataset for calculations. No defaults
        bloom_index (int, required): The bloom number for that index. In range of 0 - len(start_DOY). No defaults 
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY
    
    Returns:
        Float. Integrated chlorophyll-a value for the bloom. 
    """
    raw_data = dataset['CHL_median']
    raw_time = dataset.time.values
    bloom_start_time = np.asarray(start_DOY[bloom_index]).flatten().astype(int)
    bloom_end_time = np.asarray(end_DOY[bloom_index]).flatten().astype(int)

    lower_bound = int(bloom_start_time[0])
    upper_bound = int(bloom_end_time[0])
    bounded_time = raw_time[lower_bound:upper_bound]
    bounded_time = (bounded_time - bounded_time[0])/np.timedelta64(1,'D')
    bounded_chl = raw_data[lower_bound:upper_bound]

    integrated_chl = scipy.integrate.trapezoid(bounded_chl,bounded_time,axis=0)
    return integrated_chl

In [28]:
def yearly_bloom_chl(dataset,year,start_DOY,end_DOY):
    """
    Calculates the integrated chlorophyll for a year, considering only the chlorophyll during bloom events.

    This function isolates the yearly chlorophyll into only the chlorophyll during bloom events.
    It then integrates over the full year to get the total integrated chlorophyll in mg/m^3 * days.

    Args:
        dataset (xarray.Dataset, required): Dataset of interest. No defaults
        year (int, required): Year of interest. No defaults
        start_DOY (list, required): List of start DOYs pre-calculated from rolling_peak_window function. Defaults to start_DOY
        end_DOY (list, required): List of end DOYs pre-calculated from rolling_peak_window function. Defaults to end_DOY

    Returns:
    Float: The total integrated chlorophyll a value for the year.
    """
    start_time = np.datetime64(f"{year}-01-01")
    end_time = np.datetime64(f"{year}-12-31")
    raw_data = dataset['CHL_median'].values
    time = dataset.time.values
    total_yearly_chl = 0.0
    for i in range(len(start_DOY)):
        bloom_start_time = int(np.asarray(start_DOY[i]).flatten()[0])
        bloom_end_time = int(np.asarray(end_DOY[i]).flatten()[0])
        bloom_time = time[bloom_start_time:bloom_end_time]
        bloom_chl = raw_data[bloom_start_time:bloom_end_time]
        bloom_year = pd.to_datetime(time[bloom_start_time]).year

        if bloom_year != year and bloom_year != year-1:
            continue

        year_mask = (bloom_time >= start_time)&(bloom_time<=end_time)
        if not year_mask.any():
            continue
        
        bounded_time = bloom_time[year_mask]
        bounded_chl = bloom_chl[year_mask]
        if len(bounded_time)>1:
            bounded_time = (bounded_time - bounded_time[0])/np.timedelta64(1,'D')
            integrated_chl = scipy.integrate.trapezoid(bounded_chl,bounded_time,axis=0)
            if not np.isnan(integrated_chl):
                total_yearly_chl += integrated_chl

    return total_yearly_chl

In [29]:
def yearly_integrated_chl(dataset,year):
    """
    Calculates the annual integrated chl-a concentration (mg/m^3 * day)

    This function finds the total integrated chl-a concentration from January 1 to December 31 of the year in question.

    Args:
        dataset (xarray.Dataset, required): Raw dataset (not smoothed) for analysis. No defaults
        year (int, required): The year for calculating the integrated chlorophyll. No defaults

    Returns:
        Float. The total integrated chlorophyll for the year as one number. 
    """
    start_time = f"{year}-01-01"
    end_time = f"{year}-12-31"
    raw_data = dataset.sel(time=slice(start_time,end_time))
    chl = raw_data['CHL_median'].values
    time = raw_data['time'].values
    start_year = np.datetime64(start_time)

    bounded_time = (time - start_year)/np.timedelta64(1,'D')

    integrated_chl = scipy.integrate.trapezoid(chl,bounded_time,axis=0)
    return integrated_chl

In [30]:
def monthly_integrated_chl(dataset,year,month):
    """
    Calculates the monthly integrated chl-a concentration (mg/m^3 * day)

    This function finds the total integrated chl-a concentration from the first day to the last day of the month of the year in question.

    Args:
        dataset (xarray.Dataset, required): Raw dataset (not smoothed) for analysis. No defaults
        year (int, required): The year for calculating the integrated chlorophyll. No defaults
        month (int, required): The month number from 1-12.

    Returns:
        Float. The total integrated chlorophyll for the year as one number. 
    """
    month_string = f"{month:02d}"
    time_slice = f"{year}-{month_string}"
    raw_data = dataset.sel(time=time_slice)
    if raw_data['time'].size <= 1:
        return 0.0
    chl = raw_data['CHL_bloom_only'].values
    time = raw_data['time'].values
    start_date = np.datetime64(f"{year}-{month_string}-01")

    bounded_time = (time - start_date)/np.timedelta64(1,'D')

    integrated_chl = scipy.integrate.trapezoid(chl,bounded_time,axis=0)
    return integrated_chl

In [31]:
def percent_annual_integrated_chl(dataset,year,bloom_index,start_DOY,end_DOY):
    """
    Calculates the percentage of the annual integrated chlorophyll that one bloom makes up.
    This function uses the yearly_integrated_chl and event_integrated_chla functions.

    Args: 
        dataset (xarray.Dataset,required): Raw dataset for analysis. No defaults
        year (int, required): The year for calculating total integrated chlorophyll. No defaults
        bloom_index (int, required): The index of the bloom for calculation. No defaults
        start_DOY (list, required): The list of start DOYs for the dataset. No defaults
        end_DOY (list, required): The list of end DOYs for the dataset. No defaults
    
    Returns:
        Float. The percentage of the annual integrated chlorophyll from the bloom of interest.
    """
    annual_chl = yearly_integrated_chl(dataset,year)
    bloom_chl = event_integrated_chla(dataset,bloom_index,start_DOY=start_DOY,end_DOY=end_DOY)
    percent_of_annual = (bloom_chl/annual_chl)*100
    return percent_of_annual